# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Una foto → objetos detectados → un registro útil
## YOLO explicado para el Diplomado UTP · Cristian Alvarado
**Caso completo:** revisar una escena, localizar objetos, contar por categoría y guardar un registro CSV y una imagen marcada.

Sirve como punto de partida para revisar fotos de un parqueadero, identificar objetos de oficina o contar objetos visibles de categorías conocidas. No es un inventario certificado: los objetos ocultos o confundidos requieren revisión humana.

**Dinámica:** abrir Colab, leer la explicación, ejecutar las celdas y conversar sobre lo que muestran. No hay entregas ni ejercicios obligatorios.

1. Guarda una copia si deseas conservar cambios.
2. Ejecuta las celdas en orden. La demostración usa imágenes incluidas en la biblioteca.
3. La primera instalación y descarga de pesos necesitan internet. CPU es suficiente para estas fotos.
4. Para una foto propia, al final hay una opción de carga desactivada inicialmente.


## 1. La pregunta antes del código
**Problema:** una foto contiene varios objetos. Queremos responder: ¿qué categorías reconoce el modelo?, ¿dónde están?, ¿cuántas detecciones aparecen de cada tipo?

| Tarea de la guía | Pregunta | Salida |
|---|---|---|
| Clasificación | ¿Qué categoría tiene la imagen o un recorte? | Una etiqueta o puntuaciones |
| Detección, este caso | ¿Qué objetos aparecen y dónde? | Categoría, puntuación y caja por objeto |
| Segmentación | ¿Qué píxeles pertenecen al objeto? | Una máscara |

YOLO no es un buscador universal. Los pesos que cargamos conocen **80 categorías de COCO**; no incluyen «café sano», «café defectuoso» o todas las herramientas de una bodega. Esas clases necesitan datos etiquetados y adaptación del modelo.

**Recorrido:** foto → red preentrenada → cajas y categorías → filtro por puntuación → tabla y conteo → revisión humana.


## 2. Preparar las herramientas
Una biblioteca reúne funciones. `ultralytics` proporciona YOLO; `pandas` organiza resultados; `matplotlib` dibuja.
La instalación es preparación del entorno: todavía no entrenamos ni predecimos nada. Comprobamos la versión 8.4.175 aunque ya exista otra instalación. Si la biblioteca estaba cargada y se actualiza, será necesario reiniciar el entorno y volver a ejecutar desde esta sección. [Código oficial de esta versión](https://github.com/ultralytics/ultralytics/tree/v8.4.175).


In [ ]:
import sys, subprocess
from importlib import metadata

VERSION_ULTRALYTICS = '8.4.175'
try:
    version_instalada = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    version_instalada = None
if version_instalada != VERSION_ULTRALYTICS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'ultralytics=={VERSION_ULTRALYTICS}'])
import ultralytics
if ultralytics.__version__ != VERSION_ULTRALYTICS:
    raise RuntimeError('Ultralytics sigue cargado con otra versión. Reinicia el entorno y ejecuta de nuevo desde la preparación.')
print('Ultralytics:', ultralytics.__version__)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink
import torch
from ultralytics import YOLO
from ultralytics.utils import ASSETS

SALIDA = Path('resultados_yolo')
SALIDA.mkdir(exist_ok=True)
torch.set_num_threads(2)
TRADUCCION = {'person':'persona','bus':'bus','car':'automóvil','truck':'camión',
              'bicycle':'bicicleta','motorcycle':'motocicleta','chair':'silla',
              'bottle':'botella','cup':'taza','laptop':'portátil','cell phone':'teléfono',
              'book':'libro','backpack':'mochila','tie':'corbata'}
def nombre_es(nombre):
    return TRADUCCION.get(nombre, nombre)
print('PyTorch:',torch.__version__)
print('Los nombres no traducidos conservan la categoría original en inglés.')


## 3. Cargar lo que el modelo ya aprendió
`YOLO('yolo26n.pt')` carga pesos preentrenados de YOLO26 nano. «Nano» identifica una variante pequeña.

**Conexión con redes neuronales:** durante un entrenamiento previo se ajustaron filtros y parámetros. Ahora reutilizamos ese aprendizaje para **inferencia**, es decir, producir una predicción. En este cuaderno no ejecutamos retropropagación ni actualizamos pesos.

Los pesos se descargan automáticamente una vez. La tabla siguiente muestra qué categorías puede predecir este archivo concreto.


In [ ]:
modelo = YOLO('yolo26n.pt')
categorias = pd.DataFrame([{'id':i,'clase_original':n,'nombre':nombre_es(n)}
                           for i,n in modelo.names.items()])
print('Categorías conocidas:',len(categorias))
display(categorias.head(15))


## 4. Mirar la foto antes de pedir la predicción
Usamos `bus.jpg`, incluida en Ultralytics. Primero observa la escena sin cajas: una foto es una matriz de píxeles con tres canales de color.

**Para conversar:** ¿qué objetos visibles esperarías encontrar? Una persona parcialmente fuera del encuadre sigue siendo visible, pero puede ser más difícil de detectar.


In [ ]:
foto = ASSETS / 'bus.jpg'
original = Image.open(foto).convert('RGB')
print('Ancho y alto:',original.size,'· canales: RGB')
fig,ax=plt.subplots(figsize=(5,7))
ax.imshow(original);ax.axis('off');ax.set_title('Entrada: la foto sin predicciones')
plt.tight_layout();plt.savefig(SALIDA/'01_original.png',dpi=130);plt.show()


## 5. Detectar: entender una llamada completa
- `source`: la foto que vamos a revisar.
- `conf=0.25`: conserva detecciones con puntuación al menos 0,25.
- `imgsz=640`: referencia de tamaño para preparar la entrada; la biblioteca ajusta escala y relleno.
- `device='cpu'`: ejecuta con procesador, sin exigir GPU.

La confianza es una **puntuación del modelo**. No significa que «0,90 garantiza 90 % de aciertos». Para medir calidad necesitamos imágenes etiquetadas y comparar predicción con realidad.


In [ ]:
UMBRAL = 0.25
resultado = modelo.predict(source=str(foto), conf=UMBRAL, imgsz=640,
                           device='cpu', verbose=False)[0]
print('Detecciones conservadas:',len(resultado.boxes))
resultado.save(filename=str(SALIDA/'02_foto_detectada.jpg'))
fig,axs=plt.subplots(1,2,figsize=(10,7))
axs[0].imshow(original);axs[0].set_title('Antes: píxeles')
axs[1].imshow(resultado.plot()[:,:,::-1]);axs[1].set_title('Después: categorías, cajas y puntuaciones')
for ax in axs:ax.axis('off')
plt.tight_layout();plt.savefig(SALIDA/'03_antes_despues.png',dpi=130);plt.show()


## 6. Leer una caja y convertirla en una fila
`boxes.cls` contiene el número de categoría. `boxes.conf` contiene la puntuación. `boxes.xyxy` contiene cuatro coordenadas en la foto original.

**Ejemplo de coordenadas, solamente ilustrativo:** `[10, 20, 110, 220]` indica esquina superior izquierda en (10,20), esquina inferior derecha en (110,220), ancho 100 y alto 200 píxeles. El origen está arriba a la izquierda y el eje vertical aumenta hacia abajo.

La próxima tabla usa **las coordenadas y puntuaciones reales que devolvió YOLO**, no las del ejemplo ilustrativo.


In [ ]:
def tabla_detecciones(r):
    columnas=['deteccion','clase_id','clase_original','categoria','puntuacion',
              'x1','y1','x2','y2','ancho_px','alto_px']
    filas=[]
    for i,caja in enumerate(r.boxes, start=1):
        clase=int(caja.cls.item());nombre=r.names[clase]
        x1,y1,x2,y2=caja.xyxy[0].cpu().numpy().tolist()
        filas.append([i,clase,nombre,nombre_es(nombre),float(caja.conf.item()),
                      x1,y1,x2,y2,x2-x1,y2-y1])
    return pd.DataFrame(filas,columns=columnas)

tabla = tabla_detecciones(resultado)
display(tabla.round(3))
assert len(tabla)==len(resultado.boxes)
if not tabla.empty:
    primera=tabla.iloc[0]
    print('Primera detección:',primera.categoria,
          '· puntuación:',round(primera.puntuacion,3),
          '· tamaño:',round(primera.ancho_px),'×',round(primera.alto_px),'px')


## 7. Contar sin confundir detecciones con realidad
Agrupamos filas por categoría. Si una categoría no aparece, significa **cero detecciones conservadas**, no prueba de que el objeto no exista.

Una foto cuenta detecciones en una imagen; no cuenta personas únicas a lo largo de un video. Para eso se necesita seguimiento y una regla de conteo.


In [ ]:
def resumir(t):
    return (t.groupby('categoria').size().rename('detecciones').reset_index()
             .sort_values('detecciones',ascending=False))
resumen=resumir(tabla)
display(resumen)
fig,ax=plt.subplots(figsize=(7,3))
if not resumen.empty:
    ax.bar(resumen.categoria,resumen.detecciones,color='#0075ad')
    ax.set_yticks(range(0,int(resumen.detecciones.max())+1))
else:ax.text(.5,.5,'Sin detecciones conservadas',ha='center',transform=ax.transAxes)
ax.set_ylabel('Detecciones en esta foto');ax.set_title('Un registro derivado de las cajas')
plt.tight_layout();plt.savefig(SALIDA/'04_conteo.png',dpi=130);plt.show()


## 8. Ver qué cambia al elevar el umbral
El docente puede recorrer esta comparación ya preparada. Usamos la misma foto y los mismos pesos; solamente cambia el umbral.

Un umbral alto elimina detecciones con puntuación menor. Puede descartar errores, pero también objetos reales. **Más cajas no significa automáticamente mejor resultado.** Este conteo no es precision, recall ni mAP.


In [ ]:
umbrales=[0.15,0.50,0.85]
comparacion=[]
fig,axs=plt.subplots(1,3,figsize=(12,6))
for ax,u in zip(axs,umbrales):
    r=modelo.predict(str(foto),conf=u,imgsz=640,device='cpu',verbose=False)[0]
    ax.imshow(r.plot()[:,:,::-1]);ax.axis('off')
    ax.set_title(f'Umbral {u:.2f}: {len(r.boxes)} detecciones')
    comparacion.append({'umbral':u,'detecciones':len(r.boxes)})
plt.tight_layout();plt.savefig(SALIDA/'05_umbrales.png',dpi=130);plt.show()
display(pd.DataFrame(comparacion))


## 9. Revisar solo una categoría y acercarnos a un objeto
En un parqueadero podríamos filtrar vehículos; en este ejemplo filtramos personas. **Filtrar no vuelve a entrenar** el modelo: solo conserva categorías conocidas.

Después recortamos una caja para entender sus coordenadas. Ese acercamiento no ejecuta un clasificador adicional ni reconoce la identidad de nadie.


In [ ]:
id_persona=next(i for i,n in modelo.names.items() if n=='person')
solo_personas=modelo.predict(str(foto),classes=[id_persona],conf=UMBRAL,
                             imgsz=640,device='cpu',verbose=False)[0]
display(tabla_detecciones(solo_personas).round(3))
if len(solo_personas.boxes):
    x1,y1,x2,y2=solo_personas.boxes.xyxy[0].cpu().numpy()
    recorte=original.crop((max(0,int(x1)),max(0,int(y1)),
                           min(original.width,int(x2)),min(original.height,int(y2))))
    fig,ax=plt.subplots(figsize=(3,4));ax.imshow(recorte);ax.axis('off')
    ax.set_title('Un recorte de la primera caja de persona');plt.tight_layout();plt.show()
else:print('No hay personas conservadas con este umbral.')


## 10. Aplicarlo a una segunda foto
Usamos `zidane.jpg`, también incluida en la biblioteca. Esto comprueba el recorrido con otra entrada. Dos fotos no bastan para demostrar generalización: sirven para revisar el mecanismo.


In [ ]:
segunda=ASSETS/'zidane.jpg'
r2=modelo.predict(str(segunda),conf=UMBRAL,imgsz=640,device='cpu',verbose=False)[0]
tabla2=tabla_detecciones(r2)
display(resumir(tabla2))
fig,ax=plt.subplots(figsize=(9,5));ax.imshow(r2.plot()[:,:,::-1]);ax.axis('off')
ax.set_title('La misma herramienta, una entrada diferente');plt.tight_layout();plt.show()


## 11. Guardar un resultado que pueda usarse
Creamos una imagen marcada, una tabla por detección, un conteo por categoría y un registro de configuración. Los CSV se pueden abrir en Excel o Sheets.

La tabla organiza **predicciones**, por lo que una persona debe revisar la foto antes de convertirlas en un registro definitivo.


In [ ]:
tabla.to_csv(SALIDA/'detecciones.csv',index=False,encoding='utf-8-sig')
resumen.to_csv(SALIDA/'conteo.csv',index=False,encoding='utf-8-sig')
config={'modelo':'yolo26n.pt','ultralytics':ultralytics.__version__,
        'torch':torch.__version__,'foto':foto.name,'umbral':UMBRAL,'imgsz':640,
        'dispositivo':'cpu','entrenamiento_en_este_cuaderno':False}
(SALIDA/'configuracion.json').write_text(json.dumps(config,indent=2,ensure_ascii=False))
for p in [SALIDA/'02_foto_detectada.jpg',SALIDA/'detecciones.csv',SALIDA/'conteo.csv',SALIDA/'configuracion.json']:
    print('Archivo creado:',p)
    display(FileLink(str(p)))


## 12. Foto propia: opción para la demostración del docente
Esta sección es opcional y no pide una entrega. Con `SUBIR_FOTO = True`, Colab abre un selector. Puedes subir una foto de objetos cotidianos de las categorías conocidas. También puedes indicar una ruta de archivo ya disponible.

Los archivos se procesan en el entorno donde ejecutas el cuaderno; no se envían a una API de predicción externa.


In [ ]:
SUBIR_FOTO = False
RUTA_FOTO_PROPIA = ''  # Ejemplo: '/content/mi_oficina.jpg'
if SUBIR_FOTO:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('El selector funciona en Colab; fuera de Colab indica RUTA_FOTO_PROPIA.')
    cargados=files.upload()
    if cargados:RUTA_FOTO_PROPIA=next(iter(cargados))
if RUTA_FOTO_PROPIA:
    propia=Path(RUTA_FOTO_PROPIA)
    if not propia.is_file():raise FileNotFoundError(str(propia))
    Image.open(propia).verify()
    rp=modelo.predict(str(propia),conf=UMBRAL,imgsz=640,device='cpu',verbose=False)[0]
    tp=tabla_detecciones(rp);cp=resumir(tp)
    rp.save(filename=str(SALIDA/'mi_foto_detectada.jpg'))
    tp.to_csv(SALIDA/'mi_foto_detecciones.csv',index=False,encoding='utf-8-sig')
    cp.to_csv(SALIDA/'mi_foto_conteo.csv',index=False,encoding='utf-8-sig')
    display(cp)
    fig,ax=plt.subplots(figsize=(9,6));ax.imshow(rp.plot()[:,:,::-1]);ax.axis('off');plt.show()
else:print('Opción de foto propia desactivada: la demostración completa ya funcionó con dos fotos incluidas.')


## Conectar lo que vimos con la guía
| Bloque | Lo que acabamos de observar |
|---|---|
| 2 · Python | Bibliotecas, funciones y variables |
| 3 · Datos | Una imagen como números y resultados como tabla |
| 6 · Clasificación | Cada detección tiene una categoría; la tarea completa también localiza |
| 8 · Evaluación | La confianza no es una métrica de calidad; necesitamos verdad de referencia |
| 9 · Redes | Reutilizamos pesos aprendidos; aquí no actualizamos parámetros |
| 10 · Visión | Detección con cajas, diferente de clasificación y segmentación |
| 14 · Impacto | Un resultado numérico se transforma en un registro revisable |

**Criterio de comprensión:** reconocer la entrada y las salidas, explicar categoría/caja/puntuación, distinguir inferencia de entrenamiento e identificar por qué el conteo necesita revisión.

**Para conversar:** ¿qué objetos faltan?, ¿qué caja parece equivocada?, ¿qué cambia con el umbral?, ¿qué necesitaríamos para medir recall?, ¿el modelo conoce la categoría que necesita mi negocio?

**Para pasar a una aplicación real:** definir categorías, reunir imágenes representativas etiquetadas, separar datos, adaptar el modelo si hace falta y medir errores. Un prototipo de conteo en fotos no controla inventarios ni entradas únicas por sí solo.

**Si algo falla:** ejecuta las celdas anteriores; revisa internet para descargar pesos; confirma que el archivo es una imagen; si aparece cero detecciones, revisa categorías, foto y umbral. En caso de incompatibilidad entre torch y torchvision, instala versiones compatibles y reinicia el entorno antes de volver a ejecutar.

Fuentes oficiales: [YOLO26](https://docs.ultralytics.com/models/yolo26/), [Predicción y resultados](https://docs.ultralytics.com/modes/predict/), [Detección](https://docs.ultralytics.com/tasks/detect/). Fotos de demostración: activos incluidos en Ultralytics. Para integrar o distribuir un producto, revisa [la licencia del software y de los pesos](https://www.ultralytics.com/license).
